In [ ]:
from pyspark.sql import functions as F

In [ ]:
import os
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# Sin cuentas de storage, catalogos, esquemas ni checkpoints hard-coded.
# En Databricks Workflows se pasan como parametros de la tarea.
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("checkpoints_root", "")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
BRONZE_SCHEMA = _cfg("bronze_schema", "bronze")
STORAGE_ACCOUNT = _cfg("storage_account")
CONTAINER = _cfg("container", "landing")

if not STORAGE_ACCOUNT:
    raise ValueError(
        "Configura el widget/variable 'storage_account' (no se versiona en el repo).")

STORAGE_BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CHECKPOINTS = _cfg("checkpoints_root") or f"{STORAGE_BASE}/_checkpoints"


def bronze_table(name: str) -> str:
    return f"{CATALOG}.{BRONZE_SCHEMA}.{name}"


def checkpoint_path(name: str) -> str:
    return f"{CHECKPOINTS}/{name}"


INDEED

In [ ]:
# COMMAND ----------
# 1) INDEED (Parquet)
# Source: landing/indeed/ (tu PC sube indeed_jobs_YYYYMMDD_HHMM.parquet via azcopy)
# Dest:  job_offers.bronze.indeed

(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path('indeed')}/schema")
    .option("cloudFiles.useNotifications", "false")
    .load(f"{STORAGE_BASE}/indeed/")
    .withColumn("_ingest_date", F.current_date())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path('indeed')}/ckpt")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table("indeed"))
)

LINKEDIN

In [ ]:
# COMMAND ----------
# 2) LINKEDIN (Parquet)
# Source: landing/linkedin/ (tu PC sube jobs_YYYYMMDD_HHMMSS.parquet via azcopy)
# Dest:  job_offers.bronze.linkedin
#
# LinkedIn es el scraper que mas tarda (hasta 10h). Su output es un unico
# jobs.parquet acumulativo que el pipeline local renombra con sufijo de
# timestamp antes de subir, para que Autoloader lo detecte como archivo nuevo.

(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path('linkedin')}/schema")
    .option("cloudFiles.useNotifications", "false")
    .load(f"{STORAGE_BASE}/linkedin/")
    .withColumn("_ingest_date", F.current_date())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path('linkedin')}/ckpt")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table("linkedin"))
)

INFOJOBS

In [ ]:
# COMMAND ----------
# 3) INFOJOBS (Parquet)
# Source: landing/infojobs/ (tu PC sube offers_YYYYMMDD_HHMMSS.parquet via azcopy)
# Dest:  job_offers.bronze.infojobs
#
# InfoJobs genera muchos archivos pequenos por run (uno por ciudad/keyword).
# Autoloader los procesa todos en una sola pasada gracias al checkpoint.

(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path('infojobs')}/schema")
    .option("cloudFiles.useNotifications", "false")
    .load(f"{STORAGE_BASE}/infojobs/")
    .withColumn("_ingest_date", F.current_date())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path('infojobs')}/ckpt")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table("infojobs"))
)

Multi-Site

In [ ]:

# COMMAND ----------
# 4) MULTI_SITE (Parquet)
# Source: landing/multi_site/ (tu PC sube jobs_unified_YYYYMMDD_HHMMSS.parquet)
# Dest:  job_offers.bronze.multi_site
#
# Multi-site es un merge de 4 subscrapers (irishjobs, stepstone_nl, jobs_ch,
# glassdoor). El archivo jobs_unified.parquet se renombra con timestamp antes
# de subir para que Autoloader lo detecte.

(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path('multi_site')}/schema")
    .option("cloudFiles.useNotifications", "false")
    .load(f"{STORAGE_BASE}/multi_site/")
    .withColumn("_ingest_date", F.current_date())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path('multi_site')}/ckpt")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table("multi_site"))
)